# Praktikum 3 - HNSW

**Nama:** Radit Firansah  
**NIM:** 244107020196

Pencarian tetangga terdekat menggunakan **HNSW** (`hnswlib`), graph hierarkis.

## Percobaan 1

Membandingkan exact NN dengan HNSW pada 1000 data 2D.

In [1]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5

nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

p = hnswlib.Index(space='l2', dim=dim)
p.init_index(max_elements=num_elements, ef_construction=100, M=16)
p.add_items(data)
p.set_ef(50)

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

=== Exact NN ===
Indices: [[389 920 948 120 166]]
Distances: [[0.01261127 0.02447018 0.03513823 0.03559522 0.05287159]]
Waktu: 0.02721381187438965 detik

=== HNSW ===
Indices: [[389 920 948 120 166]]
Distances: [[0.00015904 0.00059879 0.0012347  0.00126702 0.00279541]]
Waktu: 5.221366882324219e-05 detik


## Tabel Hasil Percobaan

Metric berbeda, 1000 vs 1 juta data, dan 2D vs 5D.

In [2]:
import numpy as np
import time
import pandas as pd

import hnswlib
from sklearn.neighbors import NearestNeighbors

rows = []
for space in ['l2', 'ip']:
    for n_data in [1000, 1_000_000]:
        for dim in [2, 5]:
            np.random.seed(42)
            data = np.random.random((n_data, dim)).astype(np.float32)
            query = np.random.random((1, dim)).astype(np.float32)
            k = 5

            nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
            nn.fit(data)
            start = time.time()
            d_exact, i_exact = nn.kneighbors(query)
            t_exact = time.time() - start

            p = hnswlib.Index(space=space, dim=dim)
            p.init_index(max_elements=n_data, ef_construction=100, M=16)
            p.add_items(data)
            p.set_ef(50)
            start = time.time()
            i_hnsw, d_hnsw = p.knn_query(query, k=k)
            t_hnsw = time.time() - start

            rows.append({
                'Distance Metrics': 'L2 (Euclidean)' if space == 'l2' else 'Inner Product',
                'Jumlah data': n_data,
                'Dimensi': dim,
                'Hasil Index ENN vs ANN': f"{list(i_exact[0])} vs {list(i_hnsw[0])}",
                'Waktu ENN (ms)': round(t_exact * 1000, 4),
                'Waktu ANN (ms)': round(t_hnsw * 1000, 4),
            })

df_hasil = pd.DataFrame(rows)
df_hasil

,Distance Metrics,Jumlah data,Dimensi,Hasil Index ENN vs ANN,Waktu ENN (ms),Waktu ANN (ms)
0,L2 (Euclidean),1000,2,"[np.int64(112), np.int64(535), np.int64(777), ...",0.7105,0.0257
1,L2 (Euclidean),1000,5,"[np.int64(988), np.int64(780), np.int64(27), n...",0.3741,0.0205
2,L2 (Euclidean),1000000,2,"[np.int64(132774), np.int64(119034), np.int64(...",6.0103,0.0672
3,L2 (Euclidean),1000000,5,"[np.int64(901095), np.int64(495168), np.int64(...",9.7804,0.0644
4,Inner Product,1000,2,"[np.int64(112), np.int64(535), np.int64(777), ...",0.9053,0.0269
5,Inner Product,1000,5,"[np.int64(988), np.int64(780), np.int64(27), n...",0.3724,0.0360
6,Inner Product,1000000,2,"[np.int64(132774), np.int64(119034), np.int64(...",6.1755,0.0496
7,Inner Product,1000000,5,"[np.int64(901095), np.int64(495168), np.int64(...",7.1907,0.0651
